In [2]:
%pip install pandas openpyxl sqlalchemy pymysql


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.8 MB 1.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.0/9.8 MB 1.2 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/9.8 MB 1.3 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.8 MB 1.5 MB/s eta 0:00:06
   -------- ------------------------------- 2.1/9.8 MB 1.5 MB/s eta 0:00:06



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [4]:
DB_USER = ""             # Seu usuário do MySQL
DB_PASS = ""        # Sua senha do MySQL
DB_HOST = "localhost"        # Host (ex: localhost ou 127.0.0.1)
DB_PORT = "3306"             # Porta padrão do MySQL
DB_NAME = "scrm_malware_db"  # Nome do banco de dados criado no DDL

In [5]:
# Conexão via SQLAlchemy
DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
engine = create_engine(DATABASE_URL)

In [10]:
# ============================================================
# 1. MAPEAMENTO DE COLUNAS DA PLANILHA
# ============================================================
LOOKUP_MAPS = {
    "codebase_types": {
        "1st Party OS/Applications": "1st Party OS/Applications",
        "1st Party Firmware": "1st Party Firmware",
        "3rd Party Application": "3rd Party Application",
        "3rd Party Firmware": "3rd Party Firmware",
        "OSS": "OSS",
        "Attacker Application": "Attacker Application",
        "Unknown/NA": "Unknown/NA"
    },
    "attack_vectors": {
        "Stolen/Purchased Certificate": "Stolen/Purchased Certificate",
        "Pre-signature Insertion": "Pre-signature Insertion",
        "Default Password Exploit": "Default Password Exploit",
        "Account Access": "Account Access",
        "Self-signed/Unsigned": "Self-signed/Unsigned",
        "Broken Signature System": "Broken Signature System",
        "Unknown, Other, or N/A": "Unknown, Other, or N/A"
    },
    "distribution_vectors": {
        "Typosquatting": "Typosquatting",
        "Hijacked Updates": "Hijacked Updates",
        "Proprietary Application Store": "Proprietary Application Store",
        "3rd Party Application Store": "3rd Party Application Store",
        "Open Source Dependency": "Open Source Dependency",
        "Worm Component": "Worm Component",
        "Hardware Component": "Hardware Component",
        "Direct Download": "Direct Download",
        "Phishing": "Phishing",
        "Development Software": "Development Software",
        "Supply Chain Service Provider": "Supply Chain Service Provider",
        "Unknown, N/A, or Other": "Unknown, N/A, or Other"
    },
    "supply_chain_potentials": {
        "Credential Theft": "Credential Theft",
        "Certificate Theft": "Certificate Theft",
        "Cryptography Error": "Cryptography Error",
        "Firmware Editing": "Firmware Editing",
        "Default Password": "Default Password",
        "Code Injection": "Code Injection",
        "N/A": "N/A"
    },
    "impact_types": {
        "Data Extraction": "Data Extraction",
        "Physical Systems": "Physical Systems",
        "Backdoor Access": "Backdoor Access",
        "Cryptominer": "Cryptominer",
        "Remote Code Execution/Download": "Remote Code Execution/Download",
        "Adware": "Adware",
        "Payment Diversion": "Payment Diversion",
        "Establish BotNet": "Establish BotNet",
        "Data Damage": "Data Damage",
        "Unknown or Other": "Unknown or Other"
    }
}

LOOKUP_PKS = {
    "codebase_types": "codebase_type_id",
    "attack_vectors": "attack_vector_id",
    "distribution_vectors": "distribution_vector_id",
    "supply_chain_potentials": "potential_id",
    "impact_types": "impact_type_id"
}

def clean_val(val):
    """Auxiliar para tratar valores nulos / NaN"""
    if pd.isna(val) or val is None or str(val).strip().lower() in ['nan', 'null', 'n/a', 'none', '']:
        return None
    return str(val).strip()

def main():
    print("Iniciando leitura e processamento do arquivo SCRM-Database-v.3.xlsx...")
    
    # 2. Carrega a aba 'Data'
    df = pd.read_excel('SCRM-Database-v.3.xlsx', sheet_name='Data')
    
    # Remove linhas totalmente vazias ou sem o nome do incidente
    df = df.dropna(subset=['Name']).copy()
    print(f"Total de incidentes válidos encontrados: {len(df)}")

    with engine.begin() as conn:
        # ============================================================
        # 3. POVOAR / MAPEAR TABELAS DE DOMÍNIO (LOOKUPS)
        # ============================================================
        lookup_ids = {}

        for table_name, mapping in LOOKUP_MAPS.items():
            lookup_ids[table_name] = {}
            pk_col = LOOKUP_PKS[table_name]

            for col_name, db_name in mapping.items():
                conn.execute(
                    text(f"INSERT IGNORE INTO {table_name} (name) VALUES (:name)"),
                    {"name": db_name}
                )
                res = conn.execute(
                    text(f"SELECT {pk_col} FROM {table_name} WHERE name = :name"),
                    {"name": db_name}
                ).fetchone()
                
                lookup_ids[table_name][col_name] = res[0]

        print("Tabelas de domínio (lookups) inicializadas com sucesso.")

        # ============================================================
        # 4. PROCESSAR E INSERIR INCIDENTES E RELACIONAMENTOS
        # ============================================================
        inserted_count = 0

        for idx, row in df.iterrows():
            # A) Tratar Atacante (Attacker)
            attacker_name = clean_val(row.get('Attacker Name')) or "Unknown"
            attacker_type = clean_val(row.get('Attacker Type')) or "Unknown"
            
            # Normalizar enum de tipo de atacante
            valid_types = ['State', 'Criminal', 'Developer', 'Researcher', 'Other', 'Unknown', 'N/a']
            matching_type = [t for t in valid_types if t.lower() == attacker_type.lower()]
            attacker_type_enum = matching_type[0] if matching_type else 'Unknown'

            # Insere ou busca atacante
            conn.execute(
                text("""
                    INSERT INTO attackers (name, type) 
                    VALUES (:name, :type) 
                    ON DUPLICATE KEY UPDATE type = VALUES(type)
                """),
                {"name": attacker_name, "type": attacker_type_enum}
            )
            attacker_id = conn.execute(
                text("SELECT attacker_id FROM attackers WHERE name = :name"),
                {"name": attacker_name}
            ).fetchone()[0]

            # B) Tratar Dados do Incidente
            incident_name = clean_val(row.get('Name'))
            
            # Data do Evento
            event_date = None
            if pd.notna(row.get('Date')):
                try:
                    event_date = pd.to_datetime(row.get('Date')).strftime('%Y-%m-%d')
                except Exception:
                    event_date = None

            # Ano
            year_val = None
            if pd.notna(row.get('Year')):
                try:
                    year_val = int(float(row.get('Year')))
                except ValueError:
                    year_val = None

            # Tipo de Divulgação
            disc_raw = clean_val(row.get('Attack/Disclosure'))
            disclosure_type = 'Unknown'
            if disc_raw in ['Attack', 'Disclosure', 'Attack/Disclosure']:
                disclosure_type = disc_raw

            # Booleans e Métricas
            is_proprietary = True if str(row.get('Proprietary')).strip() in ['1', '1.0', 'True'] else False
            has_app_store = True if str(row.get('App Store?')).strip() in ['1', '1.0', 'True'] else False
            
            ease_score = int(row.get('Ease/Breadth of Access')) if pd.notna(row.get('Ease/Breadth of Access')) else None
            depth_score = int(row.get('Depth in Stack')) if pd.notna(row.get('Depth in Stack')) else None
            batch_num = int(row.get('Batch')) if pd.notna(row.get('Batch')) else None

            # Inserir na tabela 'incidents'
            sql_incident = text("""
                INSERT INTO incidents (
                    incident_name, event_date, year, disclosure_type, summary,
                    article_urls, affected_code, code_location_owner, downstream_target,
                    is_proprietary, has_app_store, ease_breadth_access_score,
                    depth_in_stack_score, batch_number, attacker_id
                ) VALUES (
                    :name, :event_date, :year, :disclosure_type, :summary,
                    :article_urls, :affected_code, :code_location_owner, :downstream_target,
                    :is_proprietary, :has_app_store, :ease_score,
                    :depth_score, :batch_num, :attacker_id
                )
            """)

            res_inc = conn.execute(sql_incident, {
                "name": incident_name,
                "event_date": event_date,
                "year": year_val,
                "disclosure_type": disclosure_type,
                "summary": clean_val(row.get('Summary')),
                "article_urls": clean_val(row.get('Article(s)')),
                "affected_code": clean_val(row.get('Affected Code')),
                "code_location_owner": clean_val(row.get('Code Location/Owner')),
                "downstream_target": clean_val(row.get('Downstream Target')),
                "is_proprietary": is_proprietary,
                "has_app_store": has_app_store,
                "ease_score": ease_score,
                "depth_score": depth_score,
                "batch_num": batch_num,
                "attacker_id": attacker_id
            })

            incident_id = res_inc.lastrowid

            # C) Tratar Relacionamentos N:N
            def insert_relations(mapping_key, target_table, fk_col_name):
                for col_name, cat_id in lookup_ids[mapping_key].items():
                    val = row.get(col_name)
                    if pd.notna(val) and float(val) == 1.0:
                        conn.execute(
                            text(f"INSERT IGNORE INTO {target_table} (incident_id, {fk_col_name}) VALUES (:inc_id, :cat_id)"),
                            {"inc_id": incident_id, "cat_id": cat_id}
                        )

            insert_relations("codebase_types", "incident_codebases", "codebase_type_id")
            insert_relations("attack_vectors", "incident_attack_vectors", "attack_vector_id")
            insert_relations("distribution_vectors", "incident_distribution_vectors", "distribution_vector_id")
            insert_relations("supply_chain_potentials", "incident_supply_chain_potentials", "potential_id")
            insert_relations("impact_types", "incident_impacts", "impact_type_id")

            inserted_count += 1

        print(f"Migração concluída com sucesso! Total de {inserted_count} incidentes migrados para o MySQL.")

if __name__ == "__main__":
    main()

Iniciando leitura e processamento do arquivo SCRM-Database-v.3.xlsx...
Total de incidentes válidos encontrados: 250
Tabelas de domínio (lookups) inicializadas com sucesso.
Migração concluída com sucesso! Total de 250 incidentes migrados para o MySQL.
